# Rattana Voice — โคลนเสียงพากย์ (F5-TTS-THAI) v1.1

1. เมนู **Runtime → Change runtime type → T4 GPU** แล้วกด Save
2. เมนู **Runtime → Run all**
3. พอขึ้นปุ่ม **Choose Files** ให้เลือกไฟล์ `ref_voice.wav` จากโฟลเดอร์ `D:\Rattana studio\voice-colab`
4. รอจนจบ แล้วจะดาวน์โหลดไฟล์ `rattana_vo.zip` ให้อัตโนมัติ ย้ายไปไว้ที่โฟลเดอร์ `voice-colab` แล้วบอก Claude ได้เลย

In [ ]:
#@title 1) ติดตั้ง (ประมาณ 2-3 นาที)
!pip -q install --force-reinstall --no-deps "numpy>=2.0"
!pip -q install --no-deps f5-tts-th
!pip -q install cached_path jieba librosa matplotlib pydub pypinyin soundfile torchdiffeq "tqdm>=4.65" transformers vocos "x_transformers>=1.31.14" pythainlp python-crfsuite ssg torchcodec
import numpy, torch; print("numpy", numpy.__version__); print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
#@title 2) อัปโหลดเสียงต้นแบบ (ref_voice.wav)
# อัปโหลด ref_voice.wav ผ่านไอคอน 📁 ที่แถบซ้ายก่อน แล้วรันช่องนี้
import os
from google.colab import files
REF_AUDIO = "ref_voice.wav"
if not os.path.exists(REF_AUDIO):
    REF_AUDIO = list(files.upload().keys())[0]
REF_TEXT = "ส่วนสถานที่อื่นๆ เราพบกันได้เลยนะครับ กับธงฟ้าประชารัฐ โดยรัตนมาร์ท ไทยช่วยไทย ลดค่าครองชีพ"
print('ref:', REF_AUDIO)

In [ ]:
#@title 3) บทพากย์ (แก้ข้อความได้ตรงนี้)
LINES = {
  "s1": "รัตนไพบูลย์ ค้าส่ง กระจายสินค้า ค้าปลีก",
  "s2": "เราคือผู้กระจายสินค้าอุปโภคบริโภค ด้วยสี่คลัง กับอีกหนึ่งหน้าร้าน สินค้ากว่าสองหมื่นรายการ รถส่งหกสิบคัน และทีมงานราวสามร้อยคน",
  "s3": "คลังของเราอยู่ที่ สมุทรสงคราม สุพรรณบุรี ราชบุรี และนครปฐม",
  "s4": "ทุกลังเดินทางเป็นระบบ จากซัพพลายเออร์ เข้าคลัง เซลล์ยิงออเดอร์ คลังจัดของ ขึ้นรถ ส่งถึงร้านค้า",
  "s5": "เบื้องหลังคือเว็บแอปกว่าหกสิบตัว ที่ทีมเราพัฒนาเอง และใช้จริงหน้างานทุกวัน",
  "s6": "และรัตนมาร์ท หน้าร้านของเราที่นครปฐม",
  "s7": "รัตนไพบูลย์",
}
TAKES = 2        # สร้างกี่เทคต่อประโยค (ไว้เลือกเทคที่ดีที่สุด)
MODELS = ["v1", "v2"]   # v1 = ออกเสียงไทยดีกว่า, v2 = ข้าม/ซ้ำคำน้อยกว่า

In [ ]:
#@title 4) สร้างเสียง
import os, time, zipfile, soundfile as sf
from f5_tts_th.tts import TTS
os.makedirs('out', exist_ok=True)
for m in MODELS:
    tts = TTS(model=m)
    for k, text in LINES.items():
        for n in range(1, TAKES + 1):
            t = time.time()
            wav = tts.infer(ref_audio=REF_AUDIO, ref_text=REF_TEXT, gen_text=text, step=32, cfg=2.0, speed=0.95)
            fn = f'out/{k}_{m}_take{n}.wav'
            sf.write(fn, wav, 24000)
            print(fn, f'{len(wav)/24000:.2f}s', f'({time.time()-t:.1f}s)')
with zipfile.ZipFile('rattana_vo.zip', 'w') as z:
    for f in sorted(os.listdir('out')): z.write(f'out/{f}', f)
print('done')

In [ ]:
#@title 5) ฟังตัวอย่าง + ดาวน์โหลด
from IPython.display import Audio, display
for f in sorted(os.listdir('out')):
    if f.endswith('_take1.wav'):
        print(f); display(Audio(f'out/{f}'))
files.download('rattana_vo.zip')